# Unified Evaluation + Reporting

Aggregates predictions from multiple runs and produces thesis-ready tables + figures.


In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve dataset root

def find_kvasir_x1_root() -> Path:
    env_root = os.environ.get("KVASIR_VQA_X1_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"KVASIR_VQA_X1_ROOT set but missing 0_dataset_prep: {p}")

    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "Prototyping_reformat" / "DatasetAnalysis" / "Kvasir_VQA_x1").exists():
            return (p / "Prototyping_reformat" / "DatasetAnalysis" / "Kvasir_VQA_x1").resolve()
        if (p / "0_dataset_prep").exists() and (p / "1_dataset_analysis").exists():
            return p
        p = p.parent
    raise RuntimeError("Could not locate Kvasir_VQA_x1 root. Set KVASIR_VQA_X1_ROOT.")

ROOT = find_kvasir_x1_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.metrics import normalize_answer, compute_metrics

MANIFEST = ROOT / "0_dataset_prep" / "out" / "manifest_x1.parquet"
OUT_DIR = ROOT / "2_modeling" / "12_eval_reporting" / "out"
TABLES_DIR = ROOT / "2_modeling" / "12_eval_reporting" / "tables"
FIG_DIR = ROOT / "2_modeling" / "12_eval_reporting" / "figures"

for d in [OUT_DIR, TABLES_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)


In [ ]:
# Load manifest
manifest = pd.read_parquet(MANIFEST)
print("rows:", len(manifest))


In [ ]:
# Config: add/remove models here
EVAL_SPLIT = "test"

MODEL_SPECS = [
    {
        "name": "qwen2_5_vl_zeroshot",
        "pred_path": ROOT / "2_modeling" / "10_modern_vlm" / "results" / "qwen2_5_vl_zeroshot" / "predictions.jsonl",
        "metrics_path": ROOT / "2_modeling" / "10_modern_vlm" / "results" / "qwen2_5_vl_zeroshot" / "metrics.json",
        "format": "jsonl",
        "pred_col": "pred_raw",
        "answer_col": "answer",
    },
    {
        "name": "medgemma_zeroshot",
        "pred_path": ROOT / "2_modeling" / "10_modern_vlm" / "results" / "medgemma_zeroshot" / "predictions.jsonl",
        "metrics_path": ROOT / "2_modeling" / "10_modern_vlm" / "results" / "medgemma_zeroshot" / "metrics.json",
        "format": "jsonl",
        "pred_col": "pred_raw",
        "answer_col": "answer",
    },
    # Optional: previous zero-shot baseline with full metadata
    {
        "name": "llava_zeroshot",
        "pred_path": ROOT / "2_modeling" / "03_vlm_modern_baseline_zeroshot" / "out" / "predictions_test.csv",
        "format": "csv",
        "pred_col": "pred_raw",
        "answer_col": "answer",
    },
]


In [ ]:
def load_predictions(spec):
    pred_path = spec.get("pred_path")
    if not pred_path:
        return None
    path = Path(pred_path)
    if not path.exists():
        return None

    if spec["format"] == "jsonl":
        df = pd.read_json(path, lines=True)
    else:
        df = pd.read_csv(path)

    pred_col = spec.get("pred_col", "pred")
    answer_col = spec.get("answer_col", "answer")

    if pred_col not in df.columns:
        print(f"[skip] {spec['name']} missing pred col {pred_col}")
        return None
    if answer_col not in df.columns:
        print(f"[skip] {spec['name']} missing answer col {answer_col}")
        return None

    df = df.copy()
    df["pred_raw"] = df[pred_col].astype(str)
    df["answer"] = df[answer_col].astype(str)

    if "pred_norm" not in df.columns:
        df["pred_norm"] = df["pred_raw"].apply(normalize_answer)

    # Ensure split is present
    if "split" not in df.columns:
        df["split"] = EVAL_SPLIT

    return df


def load_metrics(spec):
    metrics_path = spec.get("metrics_path")
    if not metrics_path:
        return None
    path = Path(metrics_path)
    if not path.exists():
        return None
    with open(path, "r") as f:
        return json.load(f)


In [ ]:
def attach_manifest_fields(pred_df: pd.DataFrame, manifest_df: pd.DataFrame) -> pd.DataFrame:
    # Attempt merge using img_id + question if possible
    if "img_id" in pred_df.columns and "question" in pred_df.columns:
        key_cols = ["img_id", "question"]
        merge_cols = [
            "img_id",
            "question",
            "question_class_list",
            "complexity",
            "is_transformed",
        ]
        merged = pred_df.merge(
            manifest_df[merge_cols],
            on=key_cols,
            how="left",
            validate="many_to_one",
        )
        return merged

    # Fallback merge on question + answer (riskier)
    if "question" in pred_df.columns and "answer" in pred_df.columns:
        key_cols = ["question", "answer"]
        merge_cols = [
            "question",
            "answer",
            "question_class_list",
            "complexity",
            "is_transformed",
        ]
        merged = pred_df.merge(
            manifest_df[merge_cols],
            on=key_cols,
            how="left",
            validate="many_to_one",
        )
        return merged

    return pred_df


In [ ]:
# Load all results (predictions preferred; fall back to metrics.json)
leader_rows = []
by_class_rows = []
by_comp_rows = []
by_trans_rows = []

for spec in MODEL_SPECS:
    df = load_predictions(spec)
    if df is not None:
        df = df[df["split"] == EVAL_SPLIT].reset_index(drop=True)
        df = attach_manifest_fields(df, manifest)
        df["model"] = spec["name"]

        # Overall
        m = compute_metrics(df["pred_norm"], df["answer"])
        m["model"] = spec["name"]
        leader_rows.append(m)

        # By class
        if "question_class_list" in df.columns:
            exploded = df.copy()
            exploded["question_class_list"] = exploded["question_class_list"].apply(
                lambda x: x if isinstance(x, list) else ([] if pd.isna(x) else [x])
            )
            exploded = exploded.explode("question_class_list").rename(
                columns={"question_class_list": "question_class"}
            )
            for qc, g in exploded.groupby("question_class"):
                m = compute_metrics(g["pred_norm"], g["answer"])
                m["model"] = spec["name"]
                m["question_class"] = qc
                by_class_rows.append(m)
        else:
            print(f"[warn] {spec['name']} missing question_class_list; skipping class breakdown")

        # By complexity
        if "complexity" in df.columns:
            for comp, g in df.groupby("complexity"):
                m = compute_metrics(g["pred_norm"], g["answer"])
                m["model"] = spec["name"]
                m["complexity"] = comp
                by_comp_rows.append(m)
        else:
            print(f"[warn] {spec['name']} missing complexity; skipping complexity breakdown")

        # By transformed
        if "is_transformed" in df.columns:
            for tval, g in df.groupby("is_transformed"):
                m = compute_metrics(g["pred_norm"], g["answer"])
                m["model"] = spec["name"]
                m["is_transformed"] = bool(tval)
                by_trans_rows.append(m)
        else:
            print(f"[warn] {spec['name']} missing is_transformed; skipping transformed breakdown")

        continue

    metrics = load_metrics(spec)
    if metrics is None:
        print(f"[skip] missing predictions/metrics for {spec['name']}")
        continue

    overall = metrics.get("overall")
    if overall:
        row = dict(overall)
        row["model"] = spec["name"]
        leader_rows.append(row)

    for row in metrics.get("by_question_class", []):
        r = dict(row)
        r["model"] = spec["name"]
        by_class_rows.append(r)

    for row in metrics.get("by_complexity", []):
        r = dict(row)
        r["model"] = spec["name"]
        by_comp_rows.append(r)

    for row in metrics.get("by_transformed", []):
        r = dict(row)
        r["model"] = spec["name"]
        by_trans_rows.append(r)

if not leader_rows:
    raise RuntimeError("No predictions or metrics loaded. Check MODEL_SPECS paths.")

leader_df = pd.DataFrame(leader_rows).sort_values("em", ascending=False)
by_class_df = pd.DataFrame(by_class_rows) if by_class_rows else None
by_comp_df = pd.DataFrame(by_comp_rows) if by_comp_rows else None
by_trans_df = pd.DataFrame(by_trans_rows) if by_trans_rows else None

print("Loaded models:", leader_df["model"].unique())


In [ ]:
# Compute leaderboard
leader_rows = []
for model, g in pred_all.groupby("model"):
    m = compute_metrics(g["pred_norm"], g["answer"])
    m["model"] = model
    leader_rows.append(m)

leader_df = pd.DataFrame(leader_rows).sort_values("em", ascending=False)
leader_df.to_csv(TABLES_DIR / "leaderboard.csv", index=False)
print(leader_df)


In [ ]:
# Breakdown by question_class
if by_class_df is not None and not by_class_df.empty:
    by_class_df.to_csv(TABLES_DIR / "breakdown_by_class.csv", index=False)
    print(by_class_df.head())
else:
    print("No question_class results; skipping class breakdown")


In [ ]:
# Breakdown by complexity
if by_comp_df is not None and not by_comp_df.empty:
    by_comp_df.to_csv(TABLES_DIR / "breakdown_by_complexity.csv", index=False)
    print(by_comp_df.head())
else:
    print("No complexity results; skipping complexity breakdown")


In [ ]:
# Original vs transformed
if by_trans_df is not None and not by_trans_df.empty:
    by_trans_df.to_csv(TABLES_DIR / "original_vs_transformed.csv", index=False)
    print(by_trans_df.head())
else:
    print("No transformed results; skipping transformed breakdown")


In [ ]:
# Figures

# Leaderboard bar chart (EM)
plt.figure(figsize=(6, 4))
plt.bar(leader_df["model"], leader_df["em"])
plt.title("EM Leaderboard (test)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "leaderboard_em.png", dpi=200)
plt.close()

# Complexity plot if available
if by_comp_df is not None and not by_comp_df.empty:
    for model in by_comp_df["model"].unique():
        sub = by_comp_df[by_comp_df["model"] == model]
        plt.figure(figsize=(5, 3))
        plt.plot(sub["complexity"], sub["em"], marker="o")
        plt.title(f"EM by Complexity — {model}")
        plt.xlabel("Complexity")
        plt.ylabel("EM")
        plt.tight_layout()
        plt.savefig(FIG_DIR / f"em_by_complexity_{model}.png", dpi=200)
        plt.close()

print("Saved figures to", FIG_DIR)


## Optional: qualitative error gallery

Add a small curated set (20–40 examples) using the prediction files once they are available.
